# **SOTA Air Pollution Forecasting: Beijing PM2.5 Prediction**
### **Ultra-High-Performance Feature Engineering, Huber Loss, & 5-Model SLSQP Ensemble**

**Competition Metric**: Root Mean Squared Error (RMSE)  
**Target RMSE Goal**: `< 13.00` (Aiming for `12.5 - 13.2` on Public/Private Leaderboard)  
**Environment**: Compatible with Google Colab (Free T4 GPU / CPU) & Local Python  

---

### **Step 1: Install & Import Dependencies**

In [1]:
!pip install -q lightgbm xgboost catboost scikit-learn pandas numpy torch matplotlib seaborn scipy

import os
import sys
import math
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
from scipy.optimize import minimize
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import ExtraTreesRegressor

import lightgbm as lgb
import xgboost as xgb
import catboost as cb
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

plt.style.use('ggplot')
warnings.filterwarnings('ignore')
print("All SOTA libraries imported successfully!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 6.9 MB/s eta 0:00:00
All SOTA libraries imported successfully!


### **Step 2: Google Colab Setup / File Check**

In [2]:
if 'google.colab' in sys.modules:
    from google.colab import files
    uploaded = files.upload() # Upload train_raw.csv and test.csv

for filename in ['train_raw.csv', 'test.csv']:
    if os.path.exists(filename):
        print(f"[FOUND] {filename} is ready ({os.path.getsize(filename) / (1024*1024):.2f} MB)")
    else:
        print(f"[MISSING] {filename} - Upload to root directory.")

Saving test.csv to test.csv
Saving train_raw.csv to train_raw.csv
[FOUND] train_raw.csv is ready (26.31 MB)
[FOUND] test.csv is ready (6.78 MB)


### **Step 3: Advanced Domain Feature Engineering (Relative Humidity, Coarse PM, EMAs, Wind Vectors)**

In [3]:
WD_MAP = {
    'N': 0.0, 'NNE': 22.5, 'NE': 45.0, 'ENE': 67.5,
    'E': 90.0, 'ESE': 112.5, 'SE': 135.0, 'SSE': 157.5,
    'S': 180.0, 'SSW': 202.5, 'SW': 225.0, 'WSW': 247.5,
    'W': 270.0, 'WNW': 292.5, 'NW': 315.0, 'NNW': 337.5
}

POLLUTANTS = ['PM2.5', 'PM10', 'SO2', 'NO2', 'CO', 'O3']
METEO = ['TEMP', 'PRES', 'DEWP', 'RAIN', 'WSPM']
NUM_FEATURES = POLLUTANTS + METEO

def process_station_timeseries(df_station):
    df = df_station.copy()

    if 'date' not in df.columns:
        df['date'] = pd.to_datetime(df[['year', 'month', 'day', 'hour']])
    df['dayofweek'] = df['date'].dt.dayofweek
    df['dayofyear'] = df['date'].dt.dayofyear

    df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24.0)
    df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24.0)
    df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12.0)
    df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12.0)
    df['dayofweek_sin'] = np.sin(2 * np.pi * df['dayofweek'] / 7.0)
    df['dayofweek_cos'] = np.cos(2 * np.pi * df['dayofweek'] / 7.0)
    df['dayofyear_sin'] = np.sin(2 * np.pi * df['dayofyear'] / 365.25)
    df['dayofyear_cos'] = np.cos(2 * np.pi * df['dayofyear'] / 365.25)
    df['is_heating_season'] = df['month'].isin([11, 12, 1, 2, 3]).astype(int)

    wd_degrees = df['wd'].map(WD_MAP)
    wd_rad = np.radians(wd_degrees)
    df['Wx'] = df['WSPM'] * np.sin(wd_rad)
    df['Wy'] = df['WSPM'] * np.cos(wd_rad)

    interp_cols = NUM_FEATURES + ['Wx', 'Wy']
    df[interp_cols] = df[interp_cols].interpolate(method='linear', limit_direction='both')
    df[interp_cols] = df[interp_cols].ffill().bfill()

    df['dew_point_depression'] = df['TEMP'] - df['DEWP']
    temp_c = df['TEMP']
    dew_c = df['DEWP']
    df['relative_humidity'] = 100.0 * np.exp((17.625 * dew_c)/(243.04 + dew_c) - (17.625 * temp_c)/(243.04 + temp_c))
    df['ventilation_index'] = df['WSPM'] * (df['dew_point_depression'] + 15.0)

    df['coarse_pm'] = np.maximum(df['PM10'] - df['PM2.5'], 0.0)
    df['PM25_PM10_ratio'] = df['PM2.5'] / (df['PM10'] + 1.0)
    df['coarse_ratio'] = df['coarse_pm'] / (df['PM10'] + 1.0)
    df['PM25_CO_ratio'] = df['PM2.5'] / (df['CO'] + 1.0)
    df['NO2_O3_ratio'] = df['NO2'] / (df['O3'] + 1.0)
    df['total_pollution'] = df['PM2.5'] + df['PM10'] + df['SO2'] + df['NO2'] + (df['CO'] / 1000.0) + df['O3']

    df['PM2.5_ema_3'] = df['PM2.5'].ewm(span=3, adjust=False).mean()
    df['PM2.5_ema_6'] = df['PM2.5'].ewm(span=6, adjust=False).mean()
    df['PM2.5_ema_12'] = df['PM2.5'].ewm(span=12, adjust=False).mean()

    for lag in range(1, 25):
        df[f'PM2.5_lag_{lag}'] = df['PM2.5'].shift(lag)

    for k in [1, 2, 3, 4, 6, 12, 24]:
        df[f'PM2.5_diff_{k}'] = df['PM2.5'] - df['PM2.5'].shift(k)
        df[f'PM2.5_diff_{k}_ratio'] = (df['PM2.5'] - df['PM2.5'].shift(k)) / (df['PM2.5'].shift(k) + 1.0)

    df['PM2.5_accel'] = (df['PM2.5'] - df['PM2.5'].shift(1)) - (df['PM2.5'].shift(1) - df['PM2.5'].shift(2))

    for col in ['PM10', 'coarse_pm', 'CO', 'NO2', 'SO2', 'TEMP', 'PRES', 'DEWP', 'WSPM', 'Wx', 'Wy', 'dew_point_depression', 'relative_humidity']:
        for k in [1, 3, 6]:
            df[f'{col}_diff_{k}'] = df[col] - df[col].shift(k)
            df[f'{col}_lag_{k}'] = df[col].shift(k)

    for w in [3, 6, 12, 24]:
        df[f'PM2.5_roll_mean_{w}h'] = df['PM2.5'].rolling(w, min_periods=1).mean()
        df[f'PM2.5_roll_std_{w}h'] = df['PM2.5'].rolling(w, min_periods=1).std().fillna(0)
        df[f'PM2.5_roll_min_{w}h'] = df['PM2.5'].rolling(w, min_periods=1).min()
        df[f'PM2.5_roll_max_{w}h'] = df['PM2.5'].rolling(w, min_periods=1).max()

    df['PM2.5_roll_range_24h'] = df['PM2.5_roll_max_24h'] - df['PM2.5_roll_min_24h']
    df['PM2.5_dev_mean_6h'] = df['PM2.5'] - df['PM2.5_roll_mean_6h']
    df['PM2.5_dev_mean_24h'] = df['PM2.5'] - df['PM2.5_roll_mean_24h']
    df['PM2.5_ratio_mean_24h'] = df['PM2.5'] / (df['PM2.5_roll_mean_24h'] + 1.0)

    for col in ['PM10', 'coarse_pm', 'CO', 'NO2', 'TEMP', 'WSPM', 'dew_point_depression', 'relative_humidity']:
        for w in [3, 6, 12]:
            df[f'{col}_roll_mean_{w}h'] = df[col].rolling(w, min_periods=1).mean()
            df[f'{col}_roll_std_{w}h'] = df[col].rolling(w, min_periods=1).std().fillna(0)

    return df

### **Step 4: Load Datasets & Extract 24-Hour Sliding Windows**

In [4]:
def create_train_dataset(train_csv_path):
    print(f"Loading {train_csv_path}...")
    df_raw = pd.read_csv(train_csv_path)
    df_raw['date'] = pd.to_datetime(df_raw[['year', 'month', 'day', 'hour']])
    df_raw = df_raw.sort_values(by=['station', 'date']).reset_index(drop=True)

    processed_stations = []
    for station, group in df_raw.groupby('station'):
        df_p = process_station_timeseries(group)
        df_p['target'] = df_p['PM2.5'].shift(-1)
        processed_stations.append(df_p)

    df_full = pd.concat(processed_stations, ignore_index=True)
    df_train = df_full.dropna(subset=['target', 'PM2.5_lag_24']).reset_index(drop=True)
    print(f"[SUCCESS] Training dataset ready with {len(df_train)} samples.")
    return df_train

def create_test_dataset(test_csv_path):
    print(f"Loading {test_csv_path}...")
    df_test_raw = pd.read_csv(test_csv_path)
    test_rows = []
    feature_vars = ['year', 'month', 'day', 'hour', 'PM2.5', 'PM10', 'SO2', 'NO2', 'CO', 'O3', 'TEMP', 'PRES', 'DEWP', 'RAIN', 'wd', 'WSPM']

    for idx, row in df_test_raw.iterrows():
        station = row['station']
        row_id = row['id']

        mini_records = []
        for lag in range(24, 0, -1):
            rec = {var: row[f'{var}_lag_{lag}'] for var in feature_vars}
            mini_records.append(rec)

        df_mini = pd.DataFrame(mini_records)
        df_mini['station'] = station
        df_mini_p = process_station_timeseries(df_mini)

        last_row = df_mini_p.iloc[-1].to_dict()
        last_row['id'] = row_id
        last_row['station'] = station
        test_rows.append(last_row)

    df_test_proc = pd.DataFrame(test_rows)
    print(f"[SUCCESS] Test dataset ready with {len(df_test_proc)} rows.")
    return df_test_proc

df_train = create_train_dataset('train_raw.csv')
df_test = create_test_dataset('test.csv')

Loading train_raw.csv...
[SUCCESS] Training dataset ready with 315348 samples.
Loading test.csv...
[SUCCESS] Test dataset ready with 4103 rows.


### **Step 5: Prepare Matrices & Station Target Encoding**

In [5]:
le_station = LabelEncoder()
df_train['station_cat'] = le_station.fit_transform(df_train['station'])
df_test['station_cat'] = le_station.transform(df_test['station'])

station_means = df_train.groupby('station_cat')['target'].mean().to_dict()
station_stds = df_train.groupby('station_cat')['target'].std().to_dict()
df_train['station_target_mean'] = df_train['station_cat'].map(station_means)
df_train['station_target_std'] = df_train['station_cat'].map(station_stds)
df_test['station_target_mean'] = df_test['station_cat'].map(station_means)
df_test['station_target_std'] = df_test['station_cat'].map(station_stds)

drop_cols = ['No', 'year', 'month', 'day', 'hour', 'date', 'wd', 'station', 'target', 'id']
feature_cols = [c for c in df_train.columns if c not in drop_cols]

X = df_train[feature_cols].copy().fillna(0)
y = df_train['target'].copy()
X_test = df_test[feature_cols].copy().fillna(0)

print(f"Total SOTA input features: {X.shape[1]}")
display(X.head())

Total SOTA input features: 224


,PM2.5,PM10,SO2,NO2,CO,O3,TEMP,PRES,DEWP,RAIN,...,dew_point_depression_roll_std_12h,relative_humidity_roll_mean_3h,relative_humidity_roll_std_3h,relative_humidity_roll_mean_6h,relative_humidity_roll_std_6h,relative_humidity_roll_mean_12h,relative_humidity_roll_std_12h,station_cat,station_target_mean,station_target_std
0,22.0,24.0,24.0,44.0,500.0,44.0,-0.4,1031.0,-17.6,0.0,...,3.242334,25.379651,1.133676,24.192510,2.428146,20.072015,4.756280,0,83.326321,81.405494
1,14.0,17.0,21.0,36.0,400.0,50.0,-1.0,1031.3,-17.3,0.0,...,3.312362,26.643307,1.055861,25.500963,1.742983,21.143068,4.955550,0,83.326321,81.405494
2,13.0,13.0,20.0,37.0,400.0,47.0,-1.5,1030.9,-16.9,0.0,...,3.164421,27.918074,1.942725,26.662294,1.984221,22.472406,4.986023,0,83.326321,81.405494
3,3.0,9.0,13.0,34.0,400.0,52.0,-1.4,1030.6,-17.6,0.0,...,2.825037,28.575059,1.138631,26.977355,2.023826,23.556084,4.606085,0,83.326321,81.405494
4,3.0,7.0,18.0,43.0,400.0,43.0,-1.5,1030.8,-17.7,0.0,...,2.394865,28.602512,1.113559,27.622909,1.446889,24.555910,4.075775,0,83.326321,81.405494


### **Step 6: PyTorch ResNet-1D BiLSTM Deep Learning Model**

In [6]:
class ResNet1DBlock(nn.Module):
    def __init__(self, channels):
        super(ResNet1DBlock, self).__init__()
        self.fc1 = nn.Linear(channels, channels)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(channels, channels)

    def forward(self, x):
        residual = x
        out = self.relu(self.fc1(x))
        out = self.fc2(out)
        return self.relu(out + residual)

class DeepResNet_BiLSTM(nn.Module):
    def __init__(self, input_dim):
        super(DeepResNet_BiLSTM, self).__init__()
        self.in_proj = nn.Linear(input_dim, 256)
        self.res1 = ResNet1DBlock(256)
        self.res2 = ResNet1DBlock(256)

        self.bilstm = nn.LSTM(256, 128, num_layers=2, batch_first=True, bidirectional=True, dropout=0.25)
        self.head = nn.Sequential(
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.15),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        h = self.in_proj(x)
        h = self.res1(h)
        h = self.res2(h)
        h = h.unsqueeze(1)
        out, _ = self.bilstm(h)
        out = out.squeeze(1)
        out = self.head(out)
        return out.squeeze(-1)

### **Step 7: 5-Fold Cross-Validation Training (LightGBM, XGBoost, CatBoost, ExtraTrees, PyTorch ResNet)**

In [7]:
oof_lgb = np.zeros(len(df_train))
oof_xgb = np.zeros(len(df_train))
oof_cb  = np.zeros(len(df_train))
oof_et  = np.zeros(len(df_train))
oof_nn  = np.zeros(len(df_train))

preds_lgb = np.zeros(len(df_test))
preds_xgb = np.zeros(len(df_test))
preds_cb  = np.zeros(len(df_test))
preds_et  = np.zeros(len(df_test))
preds_nn  = np.zeros(len(df_test))

kf = KFold(n_splits=5, shuffle=True, random_state=42)

for fold, (train_idx, val_idx) in enumerate(kf.split(X, y)):
    print(f"\n--- FOLD {fold+1} / 5 ---")
    X_tr, y_tr = X.iloc[train_idx], y.iloc[train_idx]
    X_va, y_va = X.iloc[val_idx], y.iloc[val_idx]

    # 1. LightGBM (Huber Loss Objective)
    model_lgb = lgb.LGBMRegressor(
        n_estimators=3000, learning_rate=0.018, num_leaves=127, max_depth=9,
        subsample=0.8, colsample_bytree=0.65, objective='huber', alpha=12.0, random_state=42 + fold, n_jobs=-1
    )
    model_lgb.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], callbacks=[lgb.early_stopping(80, verbose=False)])
    oof_lgb[val_idx] = model_lgb.predict(X_va)
    preds_lgb += model_lgb.predict(X_test) / 5.0
    print(f"Fold {fold+1} LightGBM (Huber) RMSE: {np.sqrt(mean_squared_error(y_va, oof_lgb[val_idx])):.4f}")

    # 2. XGBoost
    model_xgb = xgb.XGBRegressor(
        n_estimators=3000, learning_rate=0.018, max_depth=8, subsample=0.8, colsample_bytree=0.65,
        gamma=0.1, reg_alpha=0.5, reg_lambda=1.5, random_state=42 + fold, n_jobs=-1, tree_method='hist'
    )
    model_xgb.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
    oof_xgb[val_idx] = model_xgb.predict(X_va)
    preds_xgb += model_xgb.predict(X_test) / 5.0
    print(f"Fold {fold+1} XGBoost RMSE:         {np.sqrt(mean_squared_error(y_va, oof_xgb[val_idx])):.4f}")

    # 3. CatBoost (Huber Loss Objective - delta=12.0)
    model_cb = cb.CatBoostRegressor(
        iterations=2500, learning_rate=0.02, depth=8, loss_function='Huber:delta=12.0', random_seed=42 + fold, verbose=False
    )
    model_cb.fit(X_tr, y_tr, eval_set=(X_va, y_va), early_stopping_rounds=80)
    oof_cb[val_idx] = model_cb.predict(X_va)
    preds_cb += model_cb.predict(X_test) / 5.0
    print(f"Fold {fold+1} CatBoost (Huber) RMSE: {np.sqrt(mean_squared_error(y_va, oof_cb[val_idx])):.4f}")

    # 4. ExtraTrees Regressor
    model_et = ExtraTreesRegressor(n_estimators=100, max_depth=16, min_samples_split=5, n_jobs=-1, random_state=42 + fold)
    model_et.fit(X_tr, y_tr)
    oof_et[val_idx] = model_et.predict(X_va)
    preds_et += model_et.predict(X_test) / 5.0
    print(f"Fold {fold+1} ExtraTrees RMSE:     {np.sqrt(mean_squared_error(y_va, oof_et[val_idx])):.4f}")

    # 5. PyTorch ResNet-1D BiLSTM
    scaler = StandardScaler()
    X_tr_sc = scaler.fit_transform(X_tr)
    X_va_sc = scaler.transform(X_va)
    X_te_sc = scaler.transform(X_test)

    ds_tr = TensorDataset(torch.tensor(X_tr_sc, dtype=torch.float32), torch.tensor(y_tr.values, dtype=torch.float32))
    ds_va = TensorDataset(torch.tensor(X_va_sc, dtype=torch.float32), torch.tensor(y_va.values, dtype=torch.float32))

    dl_tr = DataLoader(ds_tr, batch_size=512, shuffle=True)
    dl_va = DataLoader(ds_va, batch_size=1024, shuffle=False)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    net = DeepResNet_BiLSTM(X_tr.shape[1]).to(device)
    optimizer = optim.AdamW(net.parameters(), lr=1e-3, weight_decay=1e-4)
    criterion = nn.SmoothL1Loss()

    best_loss = float('inf')
    best_preds = None
    for epoch in range(15):
        net.train()
        for bx, by in dl_tr:
            bx, by = bx.to(device), by.to(device)
            optimizer.zero_grad()
            loss = criterion(net(bx), by)
            loss.backward()
            optimizer.step()

        net.eval()
        val_preds_list = []
        with torch.no_grad():
            for bx, by in dl_va:
                val_preds_list.append(net(bx.to(device)).cpu().numpy())
        val_preds_arr = np.concatenate(val_preds_list)
        val_rmse = np.sqrt(mean_squared_error(y_va, val_preds_arr))
        if val_rmse < best_loss:
            best_loss = val_rmse
            best_preds = val_preds_arr

    oof_nn[val_idx] = best_preds
    net.eval()
    with torch.no_grad():
        preds_nn += net(torch.tensor(X_te_sc, dtype=torch.float32).to(device)).cpu().numpy() / 5.0
    print(f"Fold {fold+1} PyTorch ResNet-BiLSTM RMSE: {best_loss:.4f}")

Streaming output truncated to the last 5000 lines.
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with posit

### **Step 8: Mathematical SLSQP Optimal Blending & Out-Of-Fold Evaluation**

In [8]:
oof_list = [oof_lgb, oof_xgb, oof_cb, oof_et, oof_nn]
test_preds_list = [preds_lgb, preds_xgb, preds_cb, preds_et, preds_nn]
model_names = ['LightGBM (Huber)', 'XGBoost', 'CatBoost (Huber)', 'ExtraTrees', 'PyTorch ResNet-BiLSTM']

for name, oof in zip(model_names, oof_list):
    score = np.sqrt(mean_squared_error(y, oof))
    print(f"-> {name:25s} OOF RMSE: {score:.5f}")

def optimize_blend_weights(oof_preds_list, y_true):
    num_models = len(oof_preds_list)
    def loss_func(weights):
        w = np.array(weights)
        w = w / np.sum(w)
        blend = sum(w[i] * oof_preds_list[i] for i in range(num_models))
        return np.sqrt(mean_squared_error(y_true, blend))

    init_weights = np.ones(num_models) / num_models
    bounds = [(0.0, 1.0)] * num_models
    constraints = ({'type': 'eq', 'fun': lambda w: 1.0 - sum(w)})
    res = minimize(loss_func, init_weights, method='SLSQP', bounds=bounds, constraints=constraints)
    return res.x / np.sum(res.x)

print("\nSolving for SLSQP Optimal Blending Weights...")
opt_w = optimize_blend_weights(oof_list, y.values)
for name, w in zip(model_names, opt_w):
    print(f"   Model Weight: {name:25s} = {w:.4f}")

final_oof = sum(opt_w[i] * oof_list[i] for i in range(len(oof_list)))
final_rmse = np.sqrt(mean_squared_error(y, final_oof))

print(f"\n==================================================")
print(f"==> OPTIMAL SOTA HYBRID ENSEMBLE OOF RMSE: {final_rmse:.5f}")
print(f"==================================================")

final_test_preds = sum(opt_w[i] * test_preds_list[i] for i in range(len(test_preds_list)))
final_test_preds = np.clip(final_test_preds, a_min=0.0, a_max=None)

-> LightGBM (Huber)          OOF RMSE: 16.29323
-> XGBoost                   OOF RMSE: 15.51217
-> CatBoost (Huber)          OOF RMSE: 17.07460
-> ExtraTrees                OOF RMSE: 17.17150
-> PyTorch ResNet-BiLSTM     OOF RMSE: 82.95154

Solving for SLSQP Optimal Blending Weights...
   Model Weight: LightGBM (Huber)          = 0.0000
   Model Weight: XGBoost                   = 1.0000
   Model Weight: CatBoost (Huber)          = 0.0000
   Model Weight: ExtraTrees                = 0.0000
   Model Weight: PyTorch ResNet-BiLSTM     = 0.0000

==> OPTIMAL SOTA HYBRID ENSEMBLE OOF RMSE: 15.51217


### **Step 9: Save Final Submission File**

In [9]:
sub_df = pd.DataFrame({
    'id': df_test['id'],
    'PM2.5': final_test_preds
})

sub_filename = 'submission_sota_ensemble.csv'
sub_df.to_csv(sub_filename, index=False)
print(f"[SUCCESS] Saved {len(sub_df)} predictions to '{sub_filename}'")
display(sub_df.head(10))

if 'google.colab' in sys.modules:
    from google.colab import files
    files.download(sub_filename)

[SUCCESS] Saved 4103 predictions to 'submission_sota_ensemble.csv'


,id,PM2.5
0,test_00001,188.002296
1,test_00002,161.450592
2,test_00003,412.084709
3,test_00004,76.380213
4,test_00005,50.570659
5,test_00006,115.612720
6,test_00007,10.355963
7,test_00008,8.288122
8,test_00009,43.251203
9,test_00010,125.578699


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>